<a href="https://colab.research.google.com/github/HasanAyaz058/flyrank-ml-internship/blob/main/w06_validation_audit_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Leakage Audit

## 1. Two paper findings + my methodology questions

### Finding 1 — Mature-page refresh pattern
The paper reports materially higher health and impressions for 365+ day content refreshed within 30 days.

**Methodology question:** How exactly was the refreshed group defined, and were the outcome windows separated from the refresh window? Because this is observational, could pages selected for refresh already differ in visibility, quality, or editorial attention?

### Finding 2 — AI model performance changes after age control
The paper reports that provider families lead in different age cohorts and says this does not justify a blanket winner.

**Methodology question:** What exact outcome is being compared, and are comparable content/topic populations represented across providers within each age band? Would grouped or repeated time-aware validation show whether the observed differences generalize beyond the cached snapshot?

These are constructive questions. The paper itself says headline findings prioritize direct aggregate comparisons, ML pages are exploratory, and the study is observational.


## 2. My model under an honest split — before / after

**Before:** random row validation can place pages from the same client in both train and validation.

**After:** grouped client validation keeps every client's pages on one side. The final March→April test remains untouched and time-aware.

The Week-5 lane predicts whether next-30-day impressions fall below 80% of current-30-day impressions. Logistic Regression is used here because it was the readable selected model in Week 5.


In [5]:
import os, json, numpy as np, pandas as pd, duckdb
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

SEED=42
MIN_IMPRESSIONS=100
MIN_DAYS=20
DECLINE_THRESHOLD=0.80

try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Add your Hugging Face READ token as Colab Secret HF_TOKEN and enable notebook access.") from e

con=duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL="hf://datasets/FlyRank/internship-warehouse"
CONTENT=f"read_parquet('{REL}/dim_content.parquet')"
MONTHS={m:f"read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')" for m in ["2026-01","2026-02","2026-03","2026-04"]}

for m,src in MONTHS.items():
    print(m, con.sql(f"SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM {src}").fetchone())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-01 (7890817, datetime.date(2026, 1, 1), datetime.date(2026, 1, 31))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-02 (7355108, datetime.date(2026, 2, 1), datetime.date(2026, 2, 28))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-03 (9841378, datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04 (10424730, datetime.date(2026, 4, 1), datetime.date(2026, 4, 30))


In [6]:
parts = []

for src in MONTHS.values():
    parts.append(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        gsc_impressions,
        gsc_clicks,
        CASE
            WHEN gsc_avg_position > 0 THEN gsc_avg_position
            ELSE NULL
        END AS gsc_avg_position
    FROM {src}
    WHERE gsc_data_available IS TRUE
    """)

monthly = con.sql(f"""
WITH d AS (
    {' UNION ALL '.join(parts)}
)
SELECT
    DATE_TRUNC('month', report_date) AS "month",
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    COUNT(DISTINCT report_date) AS gsc_days
FROM d
GROUP BY 1, 2, 3
""").df()

monthly["month"] = pd.to_datetime(monthly["month"]).dt.strftime("%Y-%m")


content = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    content_updated_date,
    search_volume,
    word_count,
    content_type,
    is_published,
    is_deleted
FROM {CONTENT}
""").df()


def make_snapshot(s):
    p = (pd.Period(s, "M") - 1).strftime("%Y-%m")
    n = (pd.Period(s, "M") + 1).strftime("%Y-%m")

    cur = monthly[monthly["month"] == s].rename(columns={
        "impressions": "impressions_current30",
        "clicks": "clicks_current30",
        "avg_position": "position_current30",
        "gsc_days": "gsc_days_current30"
    })

    prev = monthly[monthly["month"] == p].rename(columns={
        "impressions": "impressions_prev30",
        "clicks": "clicks_prev30",
        "avg_position": "position_prev30",
        "gsc_days": "gsc_days_prev30"
    })

    nxt = monthly[monthly["month"] == n].rename(columns={
        "impressions": "impressions_next30",
        "clicks": "clicks_next30",
        "avg_position": "position_next30",
        "gsc_days": "gsc_days_next30"
    })

    x = (
        cur
        .merge(
            prev,
            on=["client_hash_id", "content_hash_id"]
        )
        .merge(
            nxt[
                [
                    "client_hash_id",
                    "content_hash_id",
                    "impressions_next30",
                    "clicks_next30",
                    "position_next30",
                    "gsc_days_next30"
                ]
            ],
            on=["client_hash_id", "content_hash_id"]
        )
        .merge(
            content,
            on=["client_hash_id", "content_hash_id"],
            how="left"
        )
    )

    end = pd.Timestamp(s + "-01") + pd.offsets.MonthEnd(0)

    x["staleness_days"] = (
        end - pd.to_datetime(x["content_updated_date"])
    ).dt.days

    x["position_slip"] = (
        x["position_current30"] - x["position_prev30"]
    )

    x["ctr_current30"] = (
        x["clicks_current30"] /
        x["impressions_current30"].replace(0, np.nan)
    )

    x["ctr_prev30"] = (
        x["clicks_prev30"] /
        x["impressions_prev30"].replace(0, np.nan)
    )

    x = x[
        (x["is_published"] == True) &
        (x["is_deleted"] == False) &
        (x["gsc_days_prev30"] >= MIN_DAYS) &
        (x["gsc_days_current30"] >= MIN_DAYS) &
        (x["gsc_days_next30"] >= MIN_DAYS) &
        (x["impressions_prev30"] >= MIN_IMPRESSIONS) &
        (x["impressions_current30"] >= MIN_IMPRESSIONS)
    ].copy()

    x["is_declining"] = (
        x["impressions_next30"] <
        DECLINE_THRESHOLD * x["impressions_current30"]
    ).astype(int)

    x["snapshot_month"] = s

    return x


train_df = make_snapshot("2026-02")
test_df = make_snapshot("2026-03")

print("Training:", len(train_df), "March test:", len(test_df))
print(
    "Train decline rate:",
    train_df["is_declining"].mean(),
    "Test:",
    test_df["is_declining"].mean()
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training: 53140 March test: 61032
Train decline rate: 0.17753105005645464 Test: 0.5404541879669682


### Model features

Only pre-decision fields are features. Next-30-day fields are label-only.


In [7]:
feature_cols=["impressions_prev30","impressions_current30","clicks_prev30","clicks_current30",
              "position_prev30","position_current30","position_slip","ctr_prev30","ctr_current30",
              "staleness_days","search_volume","word_count"]
for c in feature_cols:
    train_df[c]=pd.to_numeric(train_df[c],errors="coerce")
    test_df[c]=pd.to_numeric(test_df[c],errors="coerce")
train_df[feature_cols]=train_df[feature_cols].replace([np.inf,-np.inf],np.nan)
test_df[feature_cols]=test_df[feature_cols].replace([np.inf,-np.inf],np.nan)

def make_model():
    return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),
                     ("scale",StandardScaler()),
                     ("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED))])


### Before: random row split

This is the weaker diagnostic comparison, not the final validation claim.


In [8]:
X=train_df[feature_cols]; y=train_df.is_declining
Xtr,Xva,ytr,yva=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
random_model=make_model().fit(Xtr,ytr)
random_prob=random_model.predict_proba(Xva)[:,1]
random_result={"split":"Random row","rows":len(yva),
"validation_clients":train_df.loc[Xva.index,"client_hash_id"].nunique(),
"average_precision":average_precision_score(yva,random_prob),
"roc_auc":roc_auc_score(yva,random_prob)}
print(pd.DataFrame([random_result]).to_string(index=False))


     split  rows  validation_clients  average_precision  roc_auc
Random row 13285                  22           0.263654 0.636527


### After: grouped client split


In [9]:
gss=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED)
tr_idx,va_idx=next(gss.split(train_df,train_df.is_declining,groups=train_df.client_hash_id))
tr=train_df.iloc[tr_idx]; va=train_df.iloc[va_idx]
group_model=make_model().fit(tr[feature_cols],tr.is_declining)
group_prob=group_model.predict_proba(va[feature_cols])[:,1]
group_result={"split":"Grouped by client","rows":len(va),
"validation_clients":va.client_hash_id.nunique(),
"average_precision":average_precision_score(va.is_declining,group_prob),
"roc_auc":roc_auc_score(va.is_declining,group_prob)}
split_comparison=pd.DataFrame([random_result,group_result])
print(split_comparison.to_string(index=False))


            split  rows  validation_clients  average_precision  roc_auc
       Random row 13285                  22           0.263654 0.636527
Grouped by client 27575                   6           0.226526 0.556620


### Final time-aware test and Week-4 baseline comparison


In [10]:
y_test=test_df.is_declining.to_numpy()
test_prob=group_model.predict_proba(test_df[feature_cols])[:,1]

stale=(test_df.staleness_days>=180).astype(int)
slipping=(test_df.position_slip>2).astype(int)
visible=(test_df.impressions_current30>=MIN_IMPRESSIONS).astype(int)
baseline_score=stale*slipping*visible*test_df.impressions_current30

def p_at(scores,k):
    order=np.argsort(-np.asarray(scores),kind="stable")
    return float(y_test[order[:k]].mean())

final_comparison=pd.DataFrame([
 {"method":"Week-4 baseline","base_rate":y_test.mean(),
  "average_precision":average_precision_score(y_test,baseline_score),
  "roc_auc":roc_auc_score(y_test,baseline_score),
  "precision_at_20":p_at(baseline_score,20),"precision_at_50":p_at(baseline_score,50)},
 {"method":"Week-5 Logistic Regression","base_rate":y_test.mean(),
  "average_precision":average_precision_score(y_test,test_prob),
  "roc_auc":roc_auc_score(y_test,test_prob),
  "precision_at_20":p_at(test_prob,20),"precision_at_50":p_at(test_prob,50)}
])
print(final_comparison.to_string(index=False))


                    method  base_rate  average_precision  roc_auc  precision_at_20  precision_at_50
           Week-4 baseline   0.540454           0.540454 0.500000             0.95             0.98
Week-5 Logistic Regression   0.540454           0.554111 0.517839             0.55             0.54


## 3. Leakage audit


In [11]:
future_or_label={"is_declining","impressions_next30","clicks_next30","position_next30","gsc_days_next30",
                 "trend_pct","trend_direction","is_declining_label"}
product_outputs={"health_score","priority_score","action_type","decision_flag"}

print("Future/label fields in features:",sorted(future_or_label.intersection(feature_cols)))
print("Product outputs in features:",sorted(product_outputs.intersection(feature_cols)))
print("Client ID used as feature:","client_hash_id" in feature_cols)

assert future_or_label.isdisjoint(feature_cols)
assert product_outputs.isdisjoint(feature_cols)
assert "client_hash_id" not in feature_cols
assert set(test_df.snapshot_month)=={"2026-03"}

print("\nLEAKAGE AUDIT: PASS")


Future/label fields in features: []
Product outputs in features: []
Client ID used as feature: False

LEAKAGE AUDIT: PASS


## 4. Real failure examples

These are decision-support errors, not proof that the model is useless. A false positive is a page predicted to decline that did not meet the chosen future label; a false negative is the reverse.


In [12]:
test_eval=test_df[["client_hash_id","content_hash_id","is_declining","impressions_prev30",
"impressions_current30","position_prev30","position_current30","position_slip",
"staleness_days","search_volume"]].copy()
test_eval["model_probability"]=test_prob
test_eval["predicted"]=(test_prob>=.5).astype(int)
test_eval["error_type"]=np.select([
(test_eval.predicted==1)&(test_eval.is_declining==0),
(test_eval.predicted==0)&(test_eval.is_declining==1)],
["false_positive","false_negative"],default="correct")
print(test_eval.error_type.value_counts().to_string())
print("\nFalse positives:")
display(test_eval[test_eval.error_type=="false_positive"].sort_values("model_probability",ascending=False).head(3))
print("\nFalse negatives:")
display(test_eval[test_eval.error_type=="false_negative"].sort_values("model_probability",ascending=False).head(3))


error_type
correct           31639
false_negative    15208
false_positive    14185

False positives:


,client_hash_id,content_hash_id,is_declining,impressions_prev30,impressions_current30,position_prev30,position_current30,position_slip,staleness_days,search_volume,model_probability,predicted,error_type
95420,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,0,167303.0,154358.0,2.923168,3.019798,0.096630,-87,90,0.992926,1,false_positive
85101,client_e547b89c05043229,content_eadb33b5df496f4a,0,94852.0,617124.0,2.559384,2.383011,-0.176374,-73,390,0.992741,1,false_positive
5805,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,0,154502.0,205045.0,4.240872,4.544203,0.303331,-72,1000,0.989996,1,false_positive



False negatives:


,client_hash_id,content_hash_id,is_declining,impressions_prev30,impressions_current30,position_prev30,position_current30,position_slip,staleness_days,search_volume,model_probability,predicted,error_type
78518,client_3ffa76342f366962,content_16fa234ada23db28,1,1141.0,605.0,5.573354,4.763244,-0.810109,-50,<NA>,0.499994,0,false_negative
66750,client_fef1a8f436438636,content_615beeab98a52514,1,2074.0,2336.0,3.392225,3.550037,0.157812,-86,70,0.499994,0,false_negative
69826,client_23a62021009f63c4,content_a697ef3eddc4e9bf,1,3749.0,5886.0,3.237373,7.621984,4.384610,-79,0,0.499991,0,false_negative


### Interpretation

The model is not a guarantee of future decline. Errors can reflect seasonality, consolidation, external demand changes, search-system changes, low-volume noise, or other factors not represented in these features.


## 5. Claim rewrite

**Too strong:** “The model predicts which pages will lose traffic and tells us which pages should be fixed.”

**Evidence-safe:** “On the evaluated March test snapshot, the model produced a measurable ranking of pages associated with the selected future-decline label. This is directional decision-support for human review; it does not establish that a page will decline, that an edit will reverse a decline, or that the result generalizes to every client or future period.”

**Validation:** “The grouped client split is stricter than a random row split because clients are held out as groups. The observed difference is a validation diagnostic, not proof of universal out-of-sample performance.”


## Self-check

- [x] Two paper findings reviewed constructively.
- [x] Methodology questions address outcome/label definition and validation/generalization.
- [x] Random row split shown as the weaker before-design.
- [x] Grouped client split shown as the stricter after-design.
- [x] Final evaluation remains time-aware.
- [x] Future/label-derived features audited.
- [x] Product outputs excluded.
- [x] Client IDs used only for grouping.
- [x] Real false-positive and false-negative examples printed.
- [x] Claims use observed/measured/directional/decision-support language.
- [ ] Run Runtime → Run all in Colab.
- [ ] Commit as `work/notebooks/w06_validation_audit.ipynb`.
